# Retail returns

## Purpose
Ingest return requests from a simulated API, validate them,
and link them to existing order lines.

## Record grain
One record represents the current state of one return request
for one order line.

## Business rules
- Each return has a unique return_id.
- The referenced order and line must exist.
- Quantity must be a positive whole number.
- Rejected requests do not consume the returnable quantity.
- Non-rejected requests together must not exceed the ordered quantity.
- updated_at must not be earlier than created_at.
- A return request does not prove that a refund occurred.
- A return request does not automatically change inventory.

In [ ]:
import re
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "retail_operations_dev")

catalog = dbutils.widgets.get("catalog").strip()

if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalog):
    raise ValueError("Invalid catalog name.")

spark.conf.set("spark.sql.session.timeZone", "UTC")

In [ ]:
spark.sql(
    f"CREATE VOLUME IF NOT EXISTS {catalog}.bronze.api_files"
)

# Returns exports will go inside this directory.
returns_source_path = (
    f"/Volumes/{catalog}/bronze/api_files/returns"
)

dbutils.fs.mkdirs(returns_source_path)

print("Upload returns JSON files to:")
print(returns_source_path)

In [ ]:
uploaded_files = dbutils.fs.ls(returns_source_path)

json_files = [
    file
    for file in uploaded_files
    if file.name.endswith(".json")
]

for file in json_files:
    print(f"{file.name} — {file.size} bytes")

if not json_files:
    raise ValueError(
        "No returns JSON files were found. "
        "Upload an export before continuing."
    )

In [ ]:
returns_state_path = (
    f"/Volumes/{catalog}/operations/"
    "streaming_checkpoints/returns_api_v1"
)

returns_schema_path = f"{returns_state_path}/schema"
returns_checkpoint_path = f"{returns_state_path}/checkpoint"

returns_target_table = f"{catalog}.bronze.returns_raw"

In [ ]:
returns_raw_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format" , "text")
    .option("cloudFiles.schemaLocation" , returns_schema_path)
    .option("pathGlobFilter" , "*.json")
    .load(returns_source_path)
    .select(
        F.col("value").alias("raw_json"),
        F.col("_metadata.file_path").alias("source_file"),
        F.current_timestamp().alias("ingested_at")
    )
)

In [ ]:
returns_ingestion_query = (
    returns_raw_stream
    .writeStream
    .format("delta")
    .option("checkpointLocation" , returns_checkpoint_path)
    .outputMode("append")
    .trigger(availableNow = True)
    .toTable(returns_target_table)
)

returns_ingestion_query.awaitTermination()

print(f"Returns ingestion completed: {returns_target_table}")

In [ ]:
saved_returns_raw = spark.table(returns_target_table)

display(
    saved_returns_raw
    .orderBy("ingested_at", "source_file")
)

display(
    saved_returns_raw
    .groupBy("source_file")
    .agg(
        F.count("*").alias("raw_records")
    )
    .orderBy("source_file")
)

In [ ]:
from pyspark.sql import types as T


# Fields inside the original return record.
return_payload_schema = T.StructType([
    T.StructField("return_id", T.StringType(), True),
    T.StructField("order_id", T.StringType(), True),
    T.StructField("line_id", T.StringType(), True),
    T.StructField("quantity", T.StringType(), True),
    T.StructField("reason", T.StringType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("created_at", T.StringType(), True),
    T.StructField("updated_at", T.StringType(), True),
])

# Structure written by extract_returns.py.

returns_envelope_schema = T.StructType([
    T.StructField(
        "return_payload",
        return_payload_schema,
        True
    ),
    T.StructField("source_system" , T.StringType(), True),
    T.StructField("extraction_id" , T.StringType(), True),
    T.StructField("extracted_at" , T.StringType(), True),
    T.StructField("_corrupt_record", T.StringType(), True),
])

parsed_returns = (
    spark.table(returns_target_table)
    .withColumn(
        "envelope",
        F.from_json(
            F.col('raw_json'),
            returns_envelope_schema,
            {
                "mode": "PERMISSIVE",
                "columnNameOfCorruptRecord": "_corrupt_record"
            }
        )
    )
     # Bring nested fields into individual columns.
    .select(
        "raw_json",
        "source_file",
        "ingested_at",

        F.col("envelope._corrupt_record")
        .alias("_corrupt_record"),

        F.col("envelope.source_system")
        .alias("source_system"),

        F.col("envelope.extraction_id")
        .alias("extraction_id"),

        F.col("envelope.extracted_at")
        .alias("extracted_at_raw"),

        F.col("envelope.return_payload.return_id")
        .alias("return_id"),

        F.col("envelope.return_payload.order_id")
        .alias("order_id"),

        F.col("envelope.return_payload.line_id")
        .alias("line_id_raw"),

        F.col("envelope.return_payload.quantity")
        .alias("quantity_raw"),

        F.col("envelope.return_payload.reason")
        .alias("reason"),

        F.col("envelope.return_payload.status")
        .alias("status"),

        F.col("envelope.return_payload.created_at")
        .alias("created_at_raw"),

        F.col("envelope.return_payload.updated_at")
        .alias("updated_at_raw")
    )
    # Convert numeric fields without failing on invalid input.
    .withColumn(
        "line_id",
        F.expr("try_cast(line_id_raw AS BIGINT)")
    )
    .withColumn(
        "quantity",
        F.expr("try_cast(quantity_raw AS BIGINT)")
    )

    # Convert timestamp strings into timestamps.
    .withColumn(
        "created_at",
        F.try_to_timestamp("created_at_raw")
    )
    .withColumn(
        "updated_at",
        F.try_to_timestamp("updated_at_raw")
    )
    .withColumn(
        "extracted_at",
        F.try_to_timestamp("extracted_at_raw")
    )
)

display(
    parsed_returns.select(
        "return_id",
        "order_id",
        "line_id",
        "quantity",
        "reason",
        "status",
        "created_at",
        "updated_at",
        "extracted_at",
        "_corrupt_record"
    )
)

display(parsed_returns.select("*"))

In [ ]:
# True when a string is neither null nor empty/whitespace.
def has_text(column_name):
    return F.coalesce(
        F.length(F.trim(F.col(column_name))) > 0,
        F.lit(False)
    )


return_rules = [
    (
        has_text("raw_json"),
        "Missing raw JSON"
    ),
    (
        F.col("_corrupt_record").isNull(),
        "Malformed JSON or incompatible JSON structure"
    ),
    (
        has_text("return_id"),
        "Missing return_id"
    ),
    (
        has_text("order_id"),
        "Missing order_id"
    ),

    # Check both the original representation and converted value.
    (
        F.col("line_id_raw").rlike(r"^[0-9]+$")
        & F.col("line_id").isNotNull()
        & (F.col("line_id") > 0),
        "line_id must be a positive whole number"
    ),
    (
        F.col("quantity_raw").rlike(r"^[0-9]+$")
        & F.col("quantity").isNotNull()
        & (F.col("quantity") > 0),
        "quantity must be a positive whole number"
    ),

    (
        has_text("reason"),
        "Missing return reason"
    ),
    (
        F.col("status").isin(
            "requested",
            "approved",
            "received",
            "rejected"
        ),
        "Missing or unsupported return status"
    ),

    (
        F.col("created_at").isNotNull(),
        "Missing or invalid created_at"
    ),
    (
        F.col("updated_at").isNotNull(),
        "Missing or invalid updated_at"
    ),
    (
        F.col("updated_at") >= F.col("created_at"),
        "updated_at must not be earlier than created_at"
    ),

    (
        F.col("source_system") == "simulated_returns_api",
        "Missing or unexpected source_system"
    ),
    (
        has_text("extraction_id"),
        "Missing extraction_id"
    ),
    (
        F.col("extracted_at").isNotNull(),
        "Missing or invalid extracted_at"
    ),
]

In [ ]:
checked_returns = (
    parsed_returns
    .withColumn(
        'rejection_reasons',
        F.filter(
            F.array(*[
                F.when(
                    ~F.coalesce(condition , F.lit(False)),
                    F.lit(reason)
                )
                for condition , reason in return_rules
            ]),
            lambda reason : reason.isNotNull()
        )
    )
    .withColumn(
        "is_valid",
        F.size("rejection_reasons") == 0
    )
)

display(
    checked_returns.select(
        "return_id",
        "order_id",
        "line_id",
        "quantity",
        "status",
        "is_valid",
        "rejection_reasons"
    )
)

In [ ]:
valid_returns = checked_returns.filter(
    F.col("is_valid")
)

rejected_returns = checked_returns.filter(
    ~F.col("is_valid")
)

display(
    rejected_returns.select(
        "return_id",
        "raw_json",
        "rejection_reasons",
        "source_file"
    )
)

# Stop before publishing if any input records failed validation.
if rejected_returns.limit(1).count() > 0:
    raise ValueError(
        "Return records failed validation. "
        "Review rejected_returns before continuing."
    )

In [ ]:
return_business_columns = [
    "order_id",
    "line_id",
    "quantity",
    "reason",
    "status",
    "created_at"
]

return_version_conflicts = (
    valid_returns
    .groupBy("return_id", "updated_at")
    .agg(
        F.countDistinct(
            F.struct(*[
                F.col(column_name)
                for column_name in return_business_columns
            ])
        ).alias("distinct_business_versions")
    )
    .filter(
        F.col("distinct_business_versions") > 1
    )
)

display(return_version_conflicts)

if return_version_conflicts.limit(1).count() > 0:
    raise ValueError(
        "Different return states have the same return_id "
        "and updated_at. Resolve these conflicts before continuing."
    )

In [ ]:
return_identity_conflicts = (
    valid_returns
    .groupBy("return_id")
    .agg(
        F.countDistinct(
            F.struct(
                "order_id",
                "line_id",
                "created_at"
            )
        ).alias("distinct_identities")
    )
    .filter(
        F.col("distinct_identities") > 1
    )
)

display(return_identity_conflicts)

if return_identity_conflicts.limit(1).count() > 0:
    raise ValueError(
        "A return_id refers to different order lines "
        "or creation timestamps."
    )

In [ ]:
from pyspark.sql.window import Window


latest_return_window = (
    Window
    .partitionBy("return_id")
    .orderBy(
        F.col("updated_at").desc(),
        F.col("extracted_at").desc(),
        F.col("ingested_at").desc(),
        F.col("source_file").asc(),
        F.col("raw_json").asc()
    )
)


latest_returns = (
    valid_returns
    .withColumn(
        "_row_number",
        F.row_number().over(latest_return_window)
    )
    .filter(
        F.col("_row_number") == 1
    )
    .select(
        "return_id",
        "order_id",
        "line_id",
        "quantity",
        "reason",
        "status",
        "created_at",
        "updated_at",
        "source_system",
        "extraction_id",
        "extracted_at",
        "source_file",
        "ingested_at"
    )
)


display(
    latest_returns.orderBy("return_id")
)

In [ ]:
return_order_lines = (
    spark.table(f"{catalog}.silver.order_items_current")
    .select(
        "order_id",
        "line_id",
        "product_id",
        F.col("quantity").alias("ordered_quantity"),
        "unit_price",
        "currency"
    )
)

# Each order/line combination must identify exactly one row.
duplicate_order_lines = (
    return_order_lines
    .groupBy("order_id", "line_id")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_order_lines.limit(1).count() > 0:
    display(duplicate_order_lines)
    raise ValueError(
        "Duplicate order lines found. "
        "Returns cannot be matched reliably."
    )

# Quantity comparisons require a valid ordered quantity.
invalid_order_quantities = return_order_lines.filter(
    F.col("ordered_quantity").isNull()
    | (F.col("ordered_quantity") <= 0)
)

if invalid_order_quantities.limit(1).count() > 0:
    display(invalid_order_quantities)
    raise ValueError("Order lines contain invalid quantities.")

In [ ]:
returns_with_orders = (
    latest_returns
    .join(
        return_order_lines.withColumn(
            "_order_line_found",
            F.lit(True)
        ),
        on=["order_id", "line_id"],
        how="left"
    )
    .withColumn(
        "order_line_missing",
        F.col("_order_line_found").isNull()
    )
    .drop("_order_line_found")
)

missing_return_order_lines = returns_with_orders.filter(
    F.col("order_line_missing")
)

display(missing_return_order_lines)

if missing_return_order_lines.limit(1).count() > 0:
    raise ValueError(
        "Some returns reference order lines that are not available "
        "in silver.order_items_current. "
        "Check the references and refresh orders before retrying."
    )

In [ ]:
# Only the latest non-rejected requests consume returnable quantity.
return_quantity_totals = (
    latest_returns
    .filter(F.col("status") != "rejected")
    .groupBy("order_id", "line_id")
    .agg(
        F.sum(
            F.col("quantity").cast("decimal(38,0)")
        ).alias("total_non_rejected_quantity")
    )
)

return_quantity_checks = (
    return_quantity_totals
    .join(
        return_order_lines.select(
            "order_id",
            "line_id",
            "ordered_quantity"
        ),
        on=["order_id", "line_id"],
        how="left"
    )
    .withColumn(
        "exceeds_ordered_quantity",
        F.col("total_non_rejected_quantity")
        > F.col("ordered_quantity")
    )
)

excess_return_quantities = return_quantity_checks.filter(
    F.col("exceeds_ordered_quantity")
)

display(return_quantity_checks)

if excess_return_quantities.limit(1).count() > 0:
    display(excess_return_quantities)
    raise ValueError(
        "Combined non-rejected return quantities exceed "
        "the ordered quantity."
    )

print("Return order references and quantity checks passed.")

In [ ]:
current_returns = latest_returns.select(
    "return_id",
    "order_id",
    "line_id",
    "quantity",
    "reason",
    "status",
    "created_at",
    "updated_at",
    "source_system",
    "extraction_id",
    "extracted_at",
    "source_file",
    "ingested_at"
)

returns_current_table = f"{catalog}.silver.returns_current"

# Create the table structure only if the table is missing.
current_returns.createOrReplaceTempView(
    "current_returns_source"
)

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {returns_current_table}
    USING DELTA
    AS
    SELECT *
    FROM current_returns_source
    WHERE 1 = 0
""")

In [ ]:
from delta.tables import DeltaTable

(
    DeltaTable.forName(spark ,returns_current_table)
    .alias('target')
    .merge(
        current_returns.alias("source") ,
        "target.return_id = source.return_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .whenNotMatchedBySourceDelete()
    .execute()
)

print(f"Saved current returns to {returns_current_table}")

In [ ]:
saved_returns_current = (
    spark.table(returns_current_table)
    .select(*current_returns.columns)
)

# Confirm one row per return.
duplicate_saved_returns = (
    saved_returns_current
    .groupBy("return_id")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_saved_returns.limit(1).count() > 0:
    raise ValueError(
        "silver.returns_current contains duplicate return IDs."
    )

# Compare all columns in both directions.
missing_or_changed_rows = current_returns.exceptAll(
    saved_returns_current
)

unexpected_rows = saved_returns_current.exceptAll(
    current_returns
)

if (
    missing_or_changed_rows.limit(1).count() > 0
    or unexpected_rows.limit(1).count() > 0
):
    raise ValueError(
        "Saved returns do not match the computed current returns."
    )

display(
    saved_returns_current.orderBy("return_id")
)

print("Silver returns verification passed.")

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE VIEW {catalog}.gold.return_details AS
    SELECT
        r.return_id,
        r.order_id,
        r.line_id,
        o.product_id,

        r.quantity AS return_quantity,
        o.quantity AS ordered_quantity,
        o.unit_price AS order_unit_price,
        o.currency,

        r.reason,
        r.status AS return_status,
        r.created_at,
        r.updated_at,

        CAST(
            r.quantity * o.unit_price
            AS DECIMAL(38, 2)
        ) AS requested_merchandise_value,

        o.order_id IS NULL AS order_line_missing

    FROM {catalog}.silver.returns_current AS r

    LEFT JOIN {catalog}.silver.order_items_current AS o
        ON r.order_id = o.order_id
        AND r.line_id = o.line_id
""")

In [ ]:
return_details = spark.table(
    f"{catalog}.gold.return_details"
)

saved_returns = spark.table(
    f"{catalog}.silver.returns_current"
)

# A join must not multiply one return into multiple rows.
duplicate_return_details = (
    return_details
    .groupBy("return_id")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_return_details.limit(1).count() > 0:
    display(duplicate_return_details)
    raise ValueError(
        "The Gold view contains multiple rows for one return."
    )

# Every saved return must appear in the view.
if return_details.count() != saved_returns.count():
    raise ValueError(
        "The Gold view row count does not match Silver returns."
    )

missing_order_lines = return_details.filter(
    F.col("order_line_missing")
)

if missing_order_lines.limit(1).count() > 0:
    display(missing_order_lines)
    raise ValueError(
        "Some Gold returns no longer have a matching order line."
    )

display(
    return_details.orderBy("return_id")
)

print("Gold return details checks passed.")